[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/electromagnetism/eddy_brake/eddy_brake.ipynb)

# Eddy-Current Braking

Eddy-current braking computes the sheet currents, Joule heating, and braking torque in a conducting disc turning beneath a localized magnet. Motion through the magnet's field drives circulating sheet currents across the triangulated disc, and the magnetic field pushes back on those currents, slowing the rotation and depositing heat in the moving material. Anisotropic fibres that conduct more readily along their direction than across it steer the current loops and alter the drag.

The disc is modelled in planar spacetime, where positions and currents are intrinsic spatial antivectors and the magnet's field across the disc is a spatial bivector. A point moves under an angular velocity bivector `spin` by their commutator `position.commutator(spin)`, and the moving material reads the magnetic field as an electric spacetime bivector `(velocity & magnetic) ^ t`, whose pairing `electric & (edge & time)` gives the electromotive force along an edge swept along the observer's worldline `time`. Composing each triangle's sheet conductivity with `(Current & time)` yields the spacetime excitation map, so charge conservation pulls fluxes back to intrinsic vertex charges via `gradient.adjugate()`, while `edge_midpoints.commutator(Bivector).adjugate()` pulls Lorentz forces back to the braking torque so `spin & torque == force & velocity`. Pulling the fixed magnet into the rotating disc via `orientation << world_field` keeps the discrete operators fixed, and when self-inductance is included, sheet resistance and Biot–Savart inductance both act as symmetric forms on closed interior streamfunctions (see [Discrete 2-Tensor Fields on Triangulations](https://geometry.caltech.edu/pubs/GLBTD14.pdf), [Magnetic-field modeling with surface currents](https://arxiv.org/abs/2005.10060), and the [COMSOL magnetic-brake model](https://doc.comsol.com/6.4/doc/com.comsol.help.models.acdc.magnetic_brake/magnetic_brake.html)).

In Gibbs–Heaviside vector notation, a point at $\mathbf r$ moves at $\mathbf v = \boldsymbol\omega \times \mathbf r$ and carries sheet current $\mathbf J = \mathbf R(\theta)\,\boldsymbol\sigma_0\,\mathbf R(\theta)^\top(-\nabla\phi + \mathbf v \times \mathbf B)$ under a rotated $2\times 2$ conductivity matrix, with $\nabla\cdot(\mathbf R\boldsymbol\sigma_0\mathbf R^\top\nabla\phi) = \nabla\cdot(\mathbf R\boldsymbol\sigma_0\mathbf R^\top(\mathbf v \times \mathbf B))$ and axial braking torque $\boldsymbol\tau = \int \mathbf r \times (\mathbf J \times \mathbf B)\,dA$. With self-inductance on an interior streamfunction $\psi$ where $\mathbf J = \nabla\psi \times \hat{\mathbf z}$, modal amplitudes satisfy $\mathbf L\dot{\mathbf a} + \mathbf R\mathbf a = \mathbf f(\theta)$ after solving $\mathbf R\mathbf u = \lambda\mathbf L\mathbf u$.

In [ ]:
# In Colab, install numga and its examples from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ https://github.com/EelcoHoogendoorn/numga/archive/HEAD.tar.gz"

In [ ]:
%matplotlib inline
from collections.abc import Callable, Iterator

import numpy as np

from numga import Algebra, NumpyContext, stack
from numga.algebras import STA2D
from numga.sparse import SparseExtensor, spdiag
from numga.topology import Topology
from examples.util import shapes
from examples.util.animation import save_figure, show_animation, show_states
from examples.electromagnetism.eddy_brake import render

# Planar spacetime of the conducting disc, with spatial directions positive and time negative.
ga = STA2D
Scalar = ga.gatype.scalar()                                                  # Scalar+
Point = ga.gatype.antivector()                                               # Antivector-
Vector = ga.gatype.vector()                                                  # Vector+
Bivector = ga.gatype.bivector()                                              # Bivector+
Rotor = ga.gatype.rotor()                                                    # Rotor+

mv = NumpyContext(ga).multivector
Spatial = Vector.derive.within(Algebra("x+y+"))                              # Vector+
Field = Bivector.derive.touching(Algebra("t-"))                              # Bivector+
Excitation = Field.derive.dual                                               # Antibivector-
Current = Spatial.derive.dual                                                # Antivector-
Amount = ga.gatype.antiscalar()                                              # Antiscalar-
Moment = ga.gatype.antibivector()                                            # Antibivector-
Conductivity = ga.gatype((Current, Field))
Resistivity = ga.gatype((Field, Current))

# Rest-frame worldline direction pairing to +1 with `mv.t` (`mv.t.dual()` carries `-1` from `t-`).
time = -mv.t.dual()                                                          # [] Point-
# Counter-clockwise quarter turn in the disc plane, taking x toward y.
quarter_turn = (mv.xy * (-np.pi / 4)).exp()                                  # [] Rotor+

## Steady currents in a turning disc

A conducting disc turning at constant angular velocity beneath a stationary magnet carries a steady current pattern in the magnet's frame. An isotropic disc is compared against radial-fibre and circumferential-fibre discs that share the same mean conductance across an annular fibre band while conducting much more strongly along their fibres than across them. Motion through the localized magnetic patch induces edge electromotive forces, while a uniform magnetic field across the whole disc would be balanced by a purely radial electrostatic potential with zero current. Pulling the resulting spacetime excitations back across the Whitney-reconstructed potential gradient determines the vertex potential, sheet currents, Joule heating, and braking torque. The term ***simplex gradient*** here means the covector orthogonal to a corner's opposite triangle edge and scaled inversely with twice the triangle area.

In [ ]:
radius = 0.1                                       # m
thickness = 0.001                                  # m
bulk_conductivity = 5.8e7                          # S/m
labels = ("Isotropic", "Radial fibres", "Circumferential fibres")
radial_conductivity = bulk_conductivity * np.array([1.0, 1.9, 0.1])          # [3]
circumferential_conductivity = bulk_conductivity * np.array([1.0, 0.1, 1.9])  # [3]
angular_speed = 1.0                                # rad/s
field_strength = 0.2                               # T
field_width = 0.02                                 # m, Gaussian standard deviation
magnet_offset = 0.055                              # m, along x
n_divisions = 16
fibre_inner_radius = 0.2 * radius
fibre_outer_radius = 0.85 * radius
material_radii = np.array([fibre_inner_radius, fibre_outer_radius, radius])  # [3]

centre = (mv.x * magnet_offset).dual()                                       # [] Point-
magnet_field = mv.xy * field_strength                                        # [] Bivector+

In [ ]:
def disc_reconstruction(topology: Topology, vertices: Point) -> SparseExtensor:
    """Whitney edge-element reconstruction maps edge voltages `Scalar+[E]` to centroid electric
    fields `Field+[F]` from the simplex gradients of `vertices` (`Point-[V]`)."""
    edge_displacements = topology.d[0] * vertices                            # Point-[E]
    triangle_edges = topology.boundary[2, 1] * edge_displacements            # [3] Point-[F]
    # Joining two oriented boundary edges against `time` gives twice each triangle's signed area.
    double_areas = (triangle_edges[0] & triangle_edges[1]) & time            # Antiscalar-[F]
    # Each corner's simplex gradient is orthogonal to its opposite edge and scales inversely with area.
    gradients = (time & triangle_edges) / double_areas                       # [3] Vector+[F]
    tails, heads = topology.corners[2, 1]                                    # [3], [3]
    spatial = (spdiag((gradients[heads] - gradients[tails]) / 3 * Scalar) * topology.oriented[2, 1]).sum(axis=0)
    return (Vector ^ mv.t)(spatial)                                          # (Field+ <- Scalar+)[F <- E]


def fibre_conductivity(face_centers: Point, principal_conductances: np.ndarray,
                       inner_radius: float, outer_radius: float,
                       mean_conductance: float) -> Conductivity:
    """Radial and circumferential fibre directions across the annulus `[inner_radius, outer_radius]`
    assemble the sheet conductivity `[...] (Current- <- Field+)[F]` from `principal_conductances`
    (`[..., 2]`), with isotropic conductance `mean_conductance` on the inner hub and outer rim."""
    radii = face_centers.antinorm()                                          # Scalar+[F]
    radial = (face_centers / radii).cast(Current)                            # Current-[F]
    tangential = quarter_turn >> radial                                      # Current-[F]
    directions = stack((radial, tangential), axis=-1)                        # [2] Current-[F]
    in_band = (radii >= inner_radius) & (radii <= outer_radius)              # [F]
    fibre_band = mv.scalar(in_band[:, None]).field()                         # Scalar+[F]
    local_conductances = mean_conductance + fibre_band * (principal_conductances - mean_conductance)  # [..., 2] Scalar+[F]
    # Each fibre direction reads the electric field along its worldsheet `directions & time`.
    return (directions * (Field & (directions & time)) * local_conductances).sum(axis=-1)  # [...] (Current- <- Field+)[F]


topology, coordinates = shapes.concentric_disk(material_radii, n_divisions)
vertices = mv(Spatial, coordinates).dual().field()                           # Point-[V]
edge_displacements = topology.d[0] * vertices                                # Point-[E]
edge_midpoints = topology.incidence[1, 0].mean(axis=0) * vertices            # Point-[E]
corners = topology.incidence[2, 0] * vertices                                # [3] Point-[F]
triangle_edges = topology.boundary[2, 1] * edge_displacements                # [3] Point-[F]
double_areas = (triangle_edges[0] & triangle_edges[1]) & time                # Antiscalar-[F]
triangle_areas = double_areas.norm() / 2                                     # Scalar+[F]
face_centers = corners.mean(axis=0)                                          # Point-[F]
reconstruction = disc_reconstruction(topology, vertices)                     # (Field+ <- Scalar+)[F <- E]

principal = thickness * np.stack((radial_conductivity, circumferential_conductivity), axis=-1)  # [3, 2]
conductivity = fibre_conductivity(face_centers, principal, fibre_inner_radius,
                                  fibre_outer_radius, thickness * bulk_conductivity)  # [3] (Current- <- Field+)[F]

In [ ]:
def steady_response(topology: Topology, edge_displacements: Point, edge_midpoints: Point,
                    triangle_areas: Scalar, reconstruction: SparseExtensor,
                    magnetic: Bivector, spin: Bivector,
                    conductivity: Conductivity) -> tuple[Scalar, Amount, Current, Scalar, Moment]:
    """Charge conservation determines the vertex potential `Scalar+[V]`, dual-edge current flux
    `Amount-[E]`, triangle sheet current `Current-[F]`, Joule heating `Scalar+[F]`, and braking
    torque `Moment-` of a disc turning at `spin` in `magnetic`."""
    V, E, F = topology.n_elements
    # Each edge midpoint moves under an angular velocity bivector by their commutator.
    motion = edge_midpoints.commutator(Bivector)                             # (Point- <- Bivector+)[E]
    velocity = motion(spin)                                                  # [...] Point-[E]
    lorentz = (velocity & magnetic) ^ mv.t                                   # [...] Field+[E]
    emf = lorentz & (edge_displacements & time)                              # [...] Scalar+[E]

    derivative = topology.d[0] * Scalar                                      # Scalar+[E <- V]
    gradient = reconstruction(derivative)                                    # (Field+ <- Scalar+)[F <- V]
    drive = reconstruction(emf)                                              # [...] Field+[F]
    # Sweeping currents along `time` forms the spacetime excitation map into `Excitation-`.
    excitation_map = triangle_areas * (Current & time)(conductivity)         # [...] (Excitation- <- Field+)[F]
    balance = gradient.adjugate()(excitation_map(gradient))                  # [...] (Amount- <- Scalar+)[V <- V]
    # Pinning vertex 0 fixes the free potential constant on the insulated disc without altering edge voltages.
    gauge_vertex = mv.scalar((np.arange(V) == 0)[:, None]).field()           # Scalar+[V]
    gauge = balance.diagonal() * gauge_vertex                                # [...] (Amount- <- Scalar+)[V]
    potential = (balance + gauge).solve(gradient.adjugate()(excitation_map(drive)))  # [...] Scalar+[V]

    voltage = emf - derivative(potential)                                    # [...] Scalar+[E]
    electric = reconstruction(voltage)                                       # [...] Field+[F]
    current = conductivity(electric)                                         # [...] Current-[F]
    flux = reconstruction.adjugate()(triangle_areas * (current & time))      # [...] Amount-[E]

    # Pairing each dual-edge flux with the edge Lorentz 1-form gives the mechanical force covector.
    force = flux & (edge_displacements & magnetic)                           # [...] Spatial+[E]
    torque = motion.adjugate()(force)                                        # [...] Moment-
    heating = triangle_areas * (electric & (current & time))                 # [...] Scalar+[F]
    return potential, flux, current, heating, torque


steady_magnetic = magnet_field * (-(edge_midpoints - centre).antinorm().squared() / (2 * field_width ** 2)).exp()  # Bivector+[E]
steady_spin = angular_speed * mv.xy                                          # [] Bivector+
steady_potential, steady_flux, steady_current, steady_heating, steady_torque = steady_response(
    topology, edge_displacements, edge_midpoints, triangle_areas, reconstruction,
    steady_magnetic, steady_spin, conductivity)

In [ ]:
def closed_currents(topology: Topology, triangle_edges: Point,
                    triangle_areas: Scalar) -> tuple[SparseExtensor, SparseExtensor]:
    """Streamfunctions vanishing on the rim embed from interior vertices via
    `Scalar+[V <- V_interior]` and map to divergence-free sheet currents via
    `(Current- <- Scalar+)[F <- V_interior]`."""
    interior = topology.interior[0]                                          # [V_interior]
    embedder = topology.reading(interior, 0).adjugate() * Scalar             # Scalar+[V <- V_interior]
    rotated_gradients = triangle_edges.cast(Current) / (2 * triangle_areas)  # [3] Current-[F]
    curl = (spdiag(rotated_gradients * Scalar) * topology.incidence[2, 0]).sum(axis=0)  # (Current- <- Scalar+)[F <- V]
    return embedder, curl(embedder)


def streamfunction_projector(topology: Topology, triangle_edges: Point, triangle_areas: Scalar,
                             conductivity: Conductivity) -> Callable[[Current], Scalar]:
    """Resistance-weighted projection maps any sheet current `[...] Current-[F]` to the vertex
    streamfunction `[...] Scalar+[V]` of the nearest closed current vanishing on the rim."""
    embedder, current = closed_currents(topology, triangle_edges, triangle_areas)
    inverse_conductance = (triangle_areas * conductivity).batch().inverse().field()  # [...] (Field+ <- Current-)[F]
    swept = (Current & time)(current)                                        # (Excitation- <- Scalar+)[F <- V_interior]
    pairing = swept.adjugate()(spdiag(triangle_areas ** 2 * inverse_conductance))  # [...] (Amount- <- Current-)[V_interior <- F]
    resistance = pairing(current)                                            # [...] (Amount- <- Scalar+)[V_interior <- V_interior]
    return lambda face_current: embedder(resistance.solve(pairing(face_current)))


steady_streams = streamfunction_projector(topology, triangle_edges, triangle_areas,
                                          conductivity)(steady_current)      # [3] Scalar+[V]
save_figure(render.steady_figure(topology, vertices, triangle_areas, steady_current, steady_streams,
                                 centre, field_width, magnet_field),
            "eddy_brake_materials");

From left to right, the panels show the isotropic, radial-fibre, and circumferential-fibre discs. Streamfunction contours trace the closed sheet-current loops over a shared power-law colour scale of current magnitude that keeps weaker return paths visible, while the dashed circle and arrowed arc mark the Gaussian magnet's width and magnetic bivector orientation. Radial fibres stretch the two return eddies inward and outward, whereas circumferential fibres spread them into broad arcs around the annulus.

In [ ]:
signed_torques = mv.xy & steady_torque                                       # [3] Scalar+
heat_rates = steady_heating.sites.sum()                                      # [3] Scalar+
for index, label in enumerate(labels):
    print(f"{label}: torque {1e3 * float(signed_torques[index].to_array()):.2f} mN m, heating {1e3 * float(heat_rates[index].to_array()):.2f} mW")

## Braking a disc with straight fibres

Releasing a straight-fibre disc at an initial spin lets the magnet's drag slow it over time. Because the fibres rotate with the material while the magnet stays fixed in space, the current loops reshape continually with angle. Pulling the world-frame magnetic field into the body frame via `body_orientation << world_field` keeps the triangulated geometry and sheet conductivity fixed throughout the trajectory, while a midpoint spin update matches each step's rotational kinetic energy loss to the Joule heat accumulated on the triangles. Poor cross-fibre conductance requires a stronger magnet here to brake over the same duration as the isotropic disc.

In [ ]:
inertia = 0.005                                    # kg m^2, disc and attached rotor
initial_speed = 1.2                                # rad/s
initial_angle = 0.0                                # rad, straight fibres initially along x
along_conductivity = bulk_conductivity * 1.9
across_conductivity = bulk_conductivity * 0.1
straight_field_strength = 0.35                     # T
elapsed = 6.0                                      # s
n_steps = 120
time_step = elapsed / n_steps
frame_duration_ms = 40
n_animation_colours = 64
n_fibre_lines = 12
n_fibre_samples = 64

straight_magnet_field = mv.xy * straight_field_strength                      # [] Bivector+

In [ ]:
def braking(topology: Topology, vertices: Point, edge_displacements: Point, edge_midpoints: Point,
            triangle_edges: Point, triangle_areas: Scalar, reconstruction: SparseExtensor,
            centre: Point, plane: Bivector, field_strength: float, field_width: float,
            conductivity: Conductivity, inertia: float, orientation: Rotor, spin: Bivector,
            heat: Scalar, time_step: float,
            n_steps: int) -> Iterator[tuple[Rotor, Bivector, Point, Scalar, Current, Scalar]]:
    """Yields `(orientation, spin, world_vertices, heat, current, streamfunction)` over `n_steps`
    midpoint braking steps beneath a fixed Gaussian magnetic patch."""
    streamfunctions = streamfunction_projector(topology, triangle_edges, triangle_areas, conductivity)

    def body_response(body_orientation: Rotor,
                      world_spin: Bivector) -> tuple[Scalar, Amount, Current, Scalar, Moment]:
        # Sample the fixed magnet at the rotated edges and pull the field back into the material frame.
        world_edges = body_orientation >> edge_midpoints                     # [...] Point-[E]
        world_field = plane * field_strength * (-(world_edges - centre).antinorm().squared() / (2 * field_width ** 2)).exp()
        magnetic = body_orientation << world_field                           # [...] Bivector+[E]
        body_spin = body_orientation << world_spin                           # [...] Bivector+
        return steady_response(topology, edge_displacements, edge_midpoints, triangle_areas,
                               reconstruction, magnetic, body_spin, conductivity)

    _, _, current, _, _ = body_response(orientation, spin)                   # [...] Current-[F]
    yield orientation, spin, orientation >> vertices, heat, current, streamfunctions(current)
    for _ in range(n_steps):
        # Predict the half-step orientation; linearity in spin gives the drag from one unit-spin solve.
        half_turn = (spin * (-time_step / 4)).exp()                          # [...] Rotor+
        midpoint_orientation = half_turn * orientation                       # [...] Rotor+
        _, _, _, unit_heating, unit_torque = body_response(midpoint_orientation, plane)
        body_plane = midpoint_orientation << plane                           # [...] Bivector+
        drag = -(body_plane & unit_torque)                                   # [...] Scalar+

        # Evaluating torque and heating at the same midpoint spin makes kinetic loss match Joule heat.
        midpoint_spin = spin / (1 + drag * time_step / (2 * inertia))        # [...] Bivector+
        heat = heat + time_step * midpoint_spin.scalar_norm_squared() * unit_heating  # [...] Scalar+[F]
        full_turn = (midpoint_spin * (-time_step / 2)).exp()                 # [...] Rotor+
        orientation = full_turn * orientation                                # [...] Rotor+
        spin = 2 * midpoint_spin - spin                                      # [...] Bivector+
        _, _, current, _, _ = body_response(orientation, spin)               # [...] Current-[F]
        yield orientation, spin, orientation >> vertices, heat, current, streamfunctions(current)


straight_topology, straight_coordinates = shapes.triangular_disk(radius, n_divisions)
straight_vertices = mv(Spatial, straight_coordinates).dual().field()         # Point-[V]
straight_edges = straight_topology.d[0] * straight_vertices                  # Point-[E]
straight_midpoints = straight_topology.incidence[1, 0].mean(axis=0) * straight_vertices  # Point-[E]
straight_triangle_edges = straight_topology.boundary[2, 1] * straight_edges  # [3] Point-[F]
straight_double_areas = (straight_triangle_edges[0] & straight_triangle_edges[1]) & time  # Antiscalar-[F]
straight_areas = straight_double_areas.norm() / 2                            # Scalar+[F]
straight_reconstruction = disc_reconstruction(straight_topology, straight_vertices)  # (Field+ <- Scalar+)[F <- E]

straight_directions = stack((mv.x.dual(), mv.y.dual()))                      # [2] Current-
straight_principal = thickness * np.array([along_conductivity, across_conductivity])  # [2]
straight_conductivity = (straight_directions * (Field & (straight_directions & time)) * straight_principal).sum(axis=-1)[None]  # [1] Current- <- Field+
straight_orientation = (mv.xy * (-initial_angle / 2)).exp()[None]            # [1] Rotor+
straight_spin = (mv.xy * initial_speed)[None]                                # [1] Bivector+
straight_heat = (straight_areas * 0)[None]                                   # [1] Scalar+[F]
parallel_guides = (render.parallel_fibres(radius, n_fibre_lines, n_fibre_samples),)

straight_stream = braking(straight_topology, straight_vertices, straight_edges, straight_midpoints,
                          straight_triangle_edges, straight_areas, straight_reconstruction, centre, mv.xy,
                          straight_field_strength, field_width, straight_conductivity, inertia,
                          straight_orientation, straight_spin, straight_heat, time_step, n_steps)
straight_states = show_states(straight_stream, render.animate_currents, straight_topology, straight_vertices,
                              straight_areas, centre, field_width, straight_magnet_field, parallel_guides,
                              np.ones((1, 1), dtype=bool), name="eddy_brake_spin_down",
                              duration_ms=frame_duration_ms, colors=n_animation_colours)

Blue chords mark the rotating high-conductivity fibre direction and the rim dot tracks the disc angle beneath the fixed magnetic patch. Whenever the fibres align radially through the magnet, strong elongated return loops form and brake the disc sharply; a quarter turn later, when the fibres cross the patch tangentially, return currents must cross the low-conductance direction and the drag momentarily weakens.

## Braking the three material discs

Releasing the isotropic, radial-fibre, and circumferential-fibre discs with the same inertia and initial spin advances all three conductivities together as one batch. Because each material's conductivity pattern is rotationally symmetric, its current distribution stays stationary in space beneath the magnet while its amplitude decays in step with the slowing spin.

In [ ]:
initial_orientation = mv.rotor().broadcast_to(conductivity.shape)            # [3] Rotor+
initial_spin = (mv.xy * initial_speed).broadcast_to(initial_orientation.shape)  # [3] Bivector+
initial_heat = (triangle_areas * 0).broadcast_to(conductivity.shape)         # [3] Scalar+[F]

fibres = (render.radial_fibres(fibre_inner_radius, fibre_outer_radius, n_fibre_lines, n_fibre_samples),
          render.circular_fibres(fibre_inner_radius, fibre_outer_radius, n_fibre_lines, n_fibre_samples))
shown = np.stack((radial_conductivity > circumferential_conductivity,
                  circumferential_conductivity > radial_conductivity), axis=-1)  # [3, 2]

materials_stream = braking(topology, vertices, edge_displacements, edge_midpoints, triangle_edges,
                           triangle_areas, reconstruction, centre, mv.xy, field_strength, field_width,
                           conductivity, inertia, initial_orientation, initial_spin, initial_heat,
                           time_step, n_steps)
braked_states = show_states(materials_stream, render.animate_currents, topology, vertices,
                            triangle_areas, centre, field_width, magnet_field, fibres, shown,
                            name="eddy_brake_materials_spin_down", duration_ms=frame_duration_ms,
                            colors=n_animation_colours)

Blue rays and rings indicate the preferred fibre direction in the second and third panels, while the isotropic disc on the left carries no fibre guides. The rim markers show the isotropic disc coming nearly to rest while both anisotropic discs keep turning, with contours and warm shading fading on a shared scale as the induced currents subside.

## Joule heat carried by the rotating material

Although the quasistatic current pattern sits stationary beneath the magnet, the dissipated Joule energy stays in the moving triangles that pass through the patch. Without thermal conduction or cooling, each material element accumulates the heat it receives on every pass, recording the history of the braking trajectory across the disc.

In [ ]:
show_animation(render.animate_heat(topology, vertices, triangle_areas, centre, field_width,
                                   magnet_field, fibres, shown, braked_states),
               "eddy_brake_materials_heat", frame_duration_ms, colors=n_animation_colours)

Warm shading shows the accumulated Joule energy per unit area carried around by each disc on a shared power-law scale. The isotropic disc concentrates intense heating along the ring swept directly under the magnet over its short stopping angle, whereas the radial-fibre and circumferential-fibre discs spread weaker heating into broader radial and annular bands over more revolutions.

## Inductive current modes of the disc

Rapid changes in spin or field cannot alter sheet currents instantaneously because their self-generated magnetic field stores energy. Every conserved current vanishing at the insulating rim comes from an interior vertex streamfunction whose values weight each triangle's rotated simplex gradients. Sheet resistance pairs the resistive electric field with each closed current, while Biot–Savart self-inductance pairs the free-space vector potential from analytic source-triangle integrals with each closed current. Solving the generalized eigenvalue problem of resistance against inductance yields inductance-orthonormal current modes that decay independently from slowest to fastest, and thickening the disc increases sheet conductance so those magnetic relaxation times lengthen.

In [ ]:
permeability = 4 * np.pi * 1e-7                    # H/m, nonmagnetic conductor in free space
n_modes = 63
n_shown_modes = 4
inductive_thickness = 0.004                        # m
n_magnetic_block_faces = 64

In [ ]:
def magnetic_coupling_blocks(corners: Point, triangle_edges: Point, permeability: float,
                             n_block_faces: int) -> Iterator[SparseExtensor]:
    """Yields `[..., F]` target-face blocks `(Spatial+ <- Current-)[F <- F]` of the planar Biot–Savart
    vector-potential integral from analytic source-triangle potential integrals and three-point target
    quadrature."""
    lengths = triangle_edges.antinorm()                                      # [3] Scalar+[F]
    double_areas = (triangle_edges[0] & triangle_edges[1]) & time            # Antiscalar-[F]
    triangle_areas = double_areas.norm() / 2                                 # Scalar+[F]
    sense = double_areas / (2 * triangle_areas)                              # Antiscalar-[F]
    # Analytic self-integration keeps the 1/r sheet kernel finite on each source triangle.
    semiperimeter = lengths.sum(axis=0) / 2                                  # Scalar+[F]
    logarithms = (semiperimeter / (semiperimeter - lengths)).log()           # [3] Scalar+[F]
    self_integral = (4 / 3) * triangle_areas.squared() * (logarithms / lengths).sum(axis=0)  # Scalar+[F]
    F = corners.batch().shape[-1]
    indices = np.arange(F)                                                   # [F]
    tails, heads = Topology(3, [[0, 1, 2]]).corners[2, 1]                    # [3], [3]
    areas = triangle_areas.batch()                                           # [F] Scalar+
    # Midpoints between corners and centroid form an exact positive quadratic quadrature rule.
    quadrature_targets = (corners.batch() + corners.mean(axis=0).batch()[None, :]) / 2  # [3, F] Point-
    for start in range(0, F, n_block_faces):
        rows = indices[start:start + n_block_faces]
        targets = quadrature_targets[:, rows]                                # [3, ...] Point-
        relative = corners - targets[..., None]                              # [3, ..., 3] Point-[F]
        distances = relative.antinorm()                                      # [3, ..., 3] Scalar+[F]
        endpoint_sum = distances[..., tails] + distances[..., heads]         # [3, ..., 3] Scalar+[F]
        side_potential = ((endpoint_sum + lengths) / (endpoint_sum - lengths)).log()  # [3, ..., 3] Scalar+[F]
        heights = (relative[..., tails] & (triangle_edges / lengths) & time) / sense  # [3, ..., 3] Scalar+[F]
        source_potential = (heights * side_potential).sum(axis=-1)           # [3, ...] Scalar+[F]
        integral = (areas[rows] * source_potential.mean(axis=0)).batch()     # [..., F] Scalar+
        integral = integral.at[np.arange(len(rows)), rows].set(self_integral.batch()[rows])
        yield SparseExtensor.from_indices(
            integral * (permeability / (4 * np.pi)) * Current.dual_inverse(),
            rows[:, None], indices[None, :], (F, F))                         # (Spatial+ <- Current-)[F <- F]


def current_modes(topology: Topology, corners: Point, triangle_edges: Point,
                  triangle_areas: Scalar, conductivity: Conductivity,
                  permeability: float,
                  n_modes: int) -> tuple[Scalar, Scalar, SparseExtensor, SparseExtensor, Resistivity]:
    """Generalized eigenmodes of sheet resistance against Biot–Savart inductance give the `n_modes`
    slowest resistive decay rates `[..., n_modes] Scalar+`, inductance-orthonormal interior
    streamfunctions `[..., n_modes] Scalar+[V_interior]`, the closed-current and vertex-embedding
    maps, and the sheet resistivity `[...] (Field+ <- Current-)[F]`."""
    embedder, current = closed_currents(topology, triangle_edges, triangle_areas)
    resistivity = conductivity.batch().inverse().field()                     # [...] (Field+ <- Current-)[F]
    swept = (Current & time)(current)                                        # (Excitation- <- Scalar+)[F <- V_interior]
    resistance = swept.adjugate()((triangle_areas * resistivity)(current))   # [...] (Amount- <- Scalar+)[V_interior <- V_interior]

    blocks = magnetic_coupling_blocks(corners, triangle_edges, permeability, n_magnetic_block_faces)
    pairing = current.adjugate()                                             # (Amount- <- Spatial+)[V_interior <- F]
    inductance = pairing(next(blocks)(current))                              # (Amount- <- Scalar+)[V_interior <- V_interior]
    for coupling in blocks:
        inductance = inductance + pairing(coupling(current))
    inductance = (inductance + inductance.adjugate()) / 2                    # (Amount- <- Scalar+)[V_interior <- V_interior]
    decay_rates, mode_shapes = resistance.eigh(inductance, n_modes)          # [..., n_modes] Scalar+, [..., n_modes] Scalar+[V_interior]
    return decay_rates, mode_shapes, current, embedder, resistivity


inductive_conductivity = (inductive_thickness / thickness) * conductivity    # [3] (Current- <- Field+)[F]
decay_rates, mode_shapes, current_map, embedder, resistivity = current_modes(
    topology, corners, triangle_edges, triangle_areas, inductive_conductivity, permeability, n_modes)
save_figure(render.modes_figure(topology, vertices, embedder(mode_shapes[:, :n_shown_modes])),
            "eddy_brake_current_modes");

From top to bottom, the rows show the four slowest-decaying current modes of the isotropic, radial-fibre, and circumferential-fibre discs. Sheet current circulates along the streamfunction contours in opposite senses around the warm and cool extrema, with radial fibres elongating the vortices along radii and circumferential fibres stretching them into concentric rings.

## Driving the current modes over a revolution

As the disc turns beneath the fixed magnet, the motional electric field drives each inductance-orthonormal mode according to the virtual work rate on that mode's sheet current. Because the forcing repeats every revolution in the material frame, projecting samples from one full turn onto a truncated Fourier series evaluates the modal drive smoothly at any rotation angle.

In [ ]:
n_harmonics = 16
inductive_field_strength = 1.0                     # T
inductive_magnet_field = mv.xy * inductive_field_strength                    # [] Bivector+

In [ ]:
def periodic_drive(face_centers: Point, triangle_areas: Scalar, mode_shapes: Scalar,
                   current_map: SparseExtensor, centre: Point, plane: Bivector,
                   field_strength: float, field_width: float, orientation: Rotor,
                   n_harmonics: int) -> Callable[[Scalar], Scalar]:
    """Fourier projection over one revolution builds a callable mapping rotation angle `[...] Scalar+`
    to modal electromotive forcing `[..., n_modes] Scalar+` up to `n_harmonics` harmonics."""
    n_samples = 2 * n_harmonics + 1
    angles = np.linspace(0, 2 * np.pi, n_samples, endpoint=False)            # [n_samples]
    turns = (plane * (-angles / 2)).exp()                                    # [n_samples] Rotor+
    orientations = turns[:, None] * orientation                              # [n_samples, ...] Rotor+
    positions = orientations >> face_centers                                 # [n_samples, ...] Point-[F]
    world_field = plane * field_strength * (-(positions - centre).antinorm().squared() / (2 * field_width ** 2)).exp()
    magnetic = orientations << world_field                                   # [n_samples, ...] Bivector+[F]
    body_plane = orientation << plane                                        # [...] Bivector+
    velocity = face_centers.commutator(body_plane)                           # [...] Point-[F]
    electric = (velocity & magnetic) ^ mv.t                                  # [n_samples, ...] Field+[F]
    swept = (Current & time)(current_map)                                    # (Excitation- <- Scalar+)[F <- V_interior]
    loads = swept.adjugate()(triangle_areas * electric)                      # [n_samples, ...] Amount-[V_interior]
    samples = (mode_shapes & loads[..., None]).sites.sum()                   # [n_samples, ..., n_modes] Scalar+

    harmonics = np.arange(1, n_harmonics + 1)                                # [n_harmonics]
    phases = angles[:, None] * harmonics                                     # [n_samples, n_harmonics]
    mean_drive = samples.mean(axis=0)                                        # [..., n_modes] Scalar+
    cosine_drive = 2 * (samples[..., None] * np.cos(phases)[:, None, None]).mean(axis=0)  # [..., n_modes, n_harmonics] Scalar+
    sine_drive = 2 * (samples[..., None] * np.sin(phases)[:, None, None]).mean(axis=0)    # [..., n_modes, n_harmonics] Scalar+

    def evaluate_drive(angle: Scalar) -> Scalar:
        harmonic_phases = angle[..., None, None] * harmonics                 # [..., 1, n_harmonics] Scalar+
        return mean_drive + (harmonic_phases.cos() * cosine_drive + harmonic_phases.sin() * sine_drive).sum(axis=-1)

    return evaluate_drive


forcing = periodic_drive(face_centers, triangle_areas, mode_shapes, current_map, centre, mv.xy,
                         inductive_field_strength, field_width, initial_orientation, n_harmonics)

## Spinning up and braking with induction

A brief half-sine motor torque pulse accelerates the three discs from rest before releasing them to inductive magnetic braking. Advancing rotor speed and modal current amplitudes together with an implicit midpoint substep couples their power exchange symmetrically, allowing magnetic energy stored in trailing current eddies to push back on the rotor as well as dissipate into Joule heat. Retaining the weighted substep modal amplitudes reconstructs the exact cross-mode Joule heating on every triangle at each display frame without forming a dense mode-pair matrix.

In [ ]:
inductive_initial_speed = 0.0                      # rad/s
inductive_target_speed = 160.0                     # rad/s, unloaded motor pulse
spinup_time = 0.008                                # s
inductive_elapsed = 0.07                           # s
n_inductive_steps = 105
inductive_time_step = inductive_elapsed / n_inductive_steps
n_substeps = 4

substep_times = (np.arange(n_inductive_steps * n_substeps) + 0.5) * inductive_time_step / n_substeps
pulse_phase = np.clip(substep_times / spinup_time, 0, 1)
pulse_acceleration = inductive_target_speed * np.pi / (2 * spinup_time) * np.sin(np.pi * pulse_phase) * (substep_times < spinup_time)
motor_torque = mv.scalar(inertia * pulse_acceleration.reshape(n_inductive_steps, n_substeps, 1, 1))  # [n_inductive_steps, n_substeps, 1] Scalar+

In [ ]:
def inductive_braking(vertices: Point, triangle_areas: Scalar, decay_rates: Scalar,
                      mode_shapes: Scalar, current_map: SparseExtensor,
                      embedder: SparseExtensor, resistivity: Resistivity,
                      forcing: Callable[[Scalar], Scalar], plane: Bivector, inertia: float,
                      orientation: Rotor, spin: Bivector, heat: Scalar, amplitudes: Scalar,
                      motor_torque: Scalar,
                      time_step: float) -> Iterator[tuple[Rotor, Bivector, Point, Scalar, Current, Scalar, Scalar, Scalar, Scalar]]:
    """Yields `(orientation, spin, world_vertices, heat, current, streamfunction, magnetic_energy,
    thermal_energy, motor_work)` across coupled midpoint substeps driven by `motor_torque`
    (`[n_steps, n_substeps, ...] Scalar+`)."""
    n_substeps = motor_torque.shape[1]
    interval = time_step / n_substeps
    angle = mv.scalar([0]).broadcast_to(orientation.shape)                   # [...] Scalar+
    thermal_energy = heat.sites.sum()                                        # [...] Scalar+
    motor_work = thermal_energy * 0                                          # [...] Scalar+
    factors = (amplitudes * 0).broadcast_to((n_substeps,) + amplitudes.shape)  # [n_substeps, ..., n_modes] Scalar+
    damping = 1 + interval * decay_rates / 2                                 # [..., n_modes] Scalar+

    def modal_state(body_orientation: Rotor, world_spin: Bivector, face_heat: Scalar,
                    modal_amplitudes: Scalar, total_heat: Scalar,
                    work: Scalar) -> tuple[Rotor, Bivector, Point, Scalar, Current, Scalar, Scalar, Scalar, Scalar]:
        world_vertices = body_orientation >> vertices                        # [...] Point-[V]
        magnetic_energy = modal_amplitudes.squared().sum(axis=-1) / 2        # [...] Scalar+
        interior = (modal_amplitudes * mode_shapes).sum(axis=-1)             # [...] Scalar+[V_interior]
        return (body_orientation, world_spin, world_vertices, face_heat, current_map(interior),
                embedder(interior), magnetic_energy, total_heat, work)

    yield modal_state(orientation, spin, heat, amplitudes, thermal_energy, motor_work)
    for step_torques in motor_torque:
        for substep, torque_drive in enumerate(step_torques):
            speed = -(plane | spin)                                          # [...] Scalar+
            drive = forcing(angle + speed * (interval / 2))                  # [..., n_modes] Scalar+
            free_amplitudes = amplitudes / damping                           # [..., n_modes] Scalar+
            driven_amplitudes = interval / 2 * drive / damping               # [..., n_modes] Scalar+
            # Eliminate midpoint modal amplitudes from angular momentum so stored magnetic energy can return to spin.
            midpoint_speed = (speed + interval / (2 * inertia) * (torque_drive - (drive * free_amplitudes).sum(axis=-1))) / (
                1 + interval / (2 * inertia) * (drive * driven_amplitudes).sum(axis=-1))  # [...] Scalar+
            midpoint_amplitudes = free_amplitudes + midpoint_speed[..., None] * driven_amplitudes  # [..., n_modes] Scalar+
            # Retaining substep amplitude factors reconstructs cross-mode face heating without a dense mode covariance.
            factors = factors.at[substep].set(np.sqrt(interval) * midpoint_amplitudes)
            thermal_energy = thermal_energy + interval * (decay_rates * midpoint_amplitudes.squared()).sum(axis=-1)  # [...] Scalar+
            motor_work = motor_work + interval * torque_drive * midpoint_speed  # [...] Scalar+
            amplitudes = 2 * midpoint_amplitudes - amplitudes                # [..., n_modes] Scalar+
            angle = angle + interval * midpoint_speed                        # [...] Scalar+
            midpoint_spin = plane * midpoint_speed                           # [...] Bivector+
            substep_turn = (midpoint_spin * (-interval / 2)).exp()           # [...] Rotor+
            orientation = substep_turn * orientation                         # [...] Rotor+
            spin = 2 * midpoint_spin - spin                                  # [...] Bivector+
        substep_currents = current_map((factors * mode_shapes).sum(axis=-1))  # [n_substeps, ...] Current-[F]
        heat = heat + triangle_areas * (resistivity(substep_currents) & (substep_currents & time)).sum(axis=0)  # [...] Scalar+[F]
        yield modal_state(orientation, spin, heat, amplitudes, thermal_energy, motor_work)


inductive_spin = (mv.xy * inductive_initial_speed).broadcast_to(initial_orientation.shape)  # [3] Bivector+
inductive_heat = (triangle_areas * 0).broadcast_to(inductive_conductivity.shape)  # [3] Scalar+[F]
initial_amplitudes = decay_rates * 0                                         # [3, n_modes] Scalar+

inductive_stream = inductive_braking(vertices, triangle_areas, decay_rates, mode_shapes, current_map,
                                     embedder, resistivity, forcing, mv.xy, inertia,
                                     initial_orientation, inductive_spin, inductive_heat,
                                     initial_amplitudes, motor_torque, inductive_time_step)
inductive_states = show_states(inductive_stream, render.animate_induction, topology, vertices,
                               triangle_areas, centre, field_width, inductive_magnet_field, fibres,
                               shown, name="eddy_brake_inductive_currents",
                               duration_ms=frame_duration_ms, colors=n_animation_colours)

On a fixed current colour scale starting from rest, currents build during the rapid motor spin-up and are dragged downstream of the magnet in the direction of rotation instead of sitting symmetric across the patch. In the isotropic disc, detached current eddies travel partway around the annulus before decaying, while the radial and circumferential fibre bands channel those trailing loops along rays and rings.

In [ ]:
show_animation(render.animate_induction_heat(topology, vertices, triangle_areas, centre, field_width,
                                             inductive_magnet_field, fibres, shown, inductive_states),
               "eddy_brake_inductive_heat", frame_duration_ms, colors=n_animation_colours)

Accumulated Joule heat per unit area wraps smoothly around the fast-spinning discs as trailing inductive eddies continue dissipating energy well past the magnet. The isotropic disc absorbs most of its kinetic energy into a bright heated ring and brakes rapidly, whereas the two fibre-steered discs heat more gradually while retaining much more of their spin.

In [ ]:
# checks
steady_power_loss = -(steady_spin & steady_torque)                           # [3] Scalar+
steady_heat_rate = steady_heating.sites.sum()                                # [3] Scalar+
assert ((steady_power_loss - steady_heat_rate).norm() < 1e-9 * steady_heat_rate).all()

# Dual-edge current fluxes conserve charge at every vertex of the insulated disc.
assert ((topology.d[0].adjugate() * steady_flux).antinorm() < 1e-8).all()

straight_energy = stack([state[3].sites.sum() + inertia / 2 * state[1].scalar_norm_squared()
                         for state in straight_states])                      # [n_steps + 1, 1] Scalar+
assert ((straight_energy - straight_energy[0]).norm() < 1e-10 * straight_energy[0]).all()

braked_energy = stack([state[3].sites.sum() + inertia / 2 * state[1].scalar_norm_squared()
                       for state in braked_states])                          # [n_steps + 1, 3] Scalar+
assert ((braked_energy - braked_energy[0]).norm() < 1e-10 * braked_energy[0]).all()

inductive_budget = stack([state[3].sites.sum() + state[6] - state[8]
                          + inertia / 2 * state[1].scalar_norm_squared()
                          for state in inductive_states])                    # [n_inductive_steps + 1, 3] Scalar+
peak_work = inductive_states[-1][8].norm().max()                             # [] Scalar+
assert ((inductive_budget - inductive_budget[0]).norm() < 1e-7 * peak_work).all()

# Face-reconstructed Joule heat matches total modal damping dissipation across all three discs.
assert ((inductive_states[-1][3].sites.sum() - inductive_states[-1][7]).norm() < 1e-9 * peak_work).all()